0. Setup

In [1]:
import fitz
import pymupdf
import pymupdf4llm

PDF_PATH = "../data/Harwick_Technical_Documents.pdf"

print("PyMuPDF:", pymupdf.__version__)

doc = fitz.open(PDF_PATH)

print("Pages:", doc.page_count)

PyMuPDF: 1.27.2.3
Pages: 159


#### 1. Visual Inspection First
Before extracting anything, look at the page.

In [2]:
page = doc[0]

pix = page.get_pixmap(dpi=200)

pix.save("page_1.png")

print("Saved page_1.png")

Saved page_1.png


#### 2. Raw Text Extraction

In [ ]:
page = doc[0]

text = page.get_text()

print(text[:5000])

#### 3. Understand PDF Blocks

A PDF is not text.

A PDF is positioned objects.

In [ ]:
blocks = page.get_text("blocks")

print("Block count:", len(blocks))

In [ ]:
for idx, block in enumerate(blocks):

    print("\n" + "=" * 80)
    print(f"BLOCK {idx}")
    print("=" * 80)

    print("BBox:", block[:4])
    print("Type:", block[6])

    print(block[4][:500])

#### 4. Reading Order Reconstruction
PDFs don't guarantee order.

PyMuPDF gives coordinates.

In [ ]:
blocks = page.get_text("blocks")

blocks = sorted(
    blocks,
    key=lambda b: (b[1], b[0])
)

for block in blocks:
    print(block[4])

#### 5. Deep Dive Into Structure

In [ ]:
content = page.get_text("dict")

In [ ]:
from pprint import pprint

pprint(content["blocks"][0])

In [ ]:
for block in content["blocks"]:

    if "lines" not in block:
        continue

    for line in block["lines"]:

        for span in line["spans"]:

            print(span["text"])
            print(span["font"])
            print(span["size"])

#### 6. Word Level Extraction

In [ ]:
words = page.get_text("words")

for word in words[:20]:
    print(word)

#### 7. Image Analysis

In [ ]:
images = page.get_images(full=True)

print("Images:", len(images))

In [ ]:
for idx, image in enumerate(images):

    xref = image[0]

    pix = fitz.Pixmap(doc, xref)

    output = f"image_{idx}.png"

    if pix.alpha:
        pix = fitz.Pixmap(fitz.csRGB, pix)

    pix.save(output)

    print(output)

#### 8. Table Detection

In [ ]:
tables = page.find_tables()

print("Detected Tables:", len(tables.tables))

In [ ]:
for idx, table in enumerate(tables.tables):

    print("\nTABLE", idx)

    data = table.extract()

    for row in data:
        print(row)

#### 9. Compare Raw Text vs Tables

In [ ]:
print(page.get_text()[:3000])

In [ ]:
table.extract()

#### 10. PyMuPDF4LLM

Now compare against the LLM-friendly version.

In [3]:
markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[0]
)

print(markdown)

=== Document parser messages ===
Using Tesseract for OCR processing.

**==> picture [415 x 83] intentionally omitted <==**

## **Table Of Content** 

|**1. **|**GTP------------------------------------------------------------------------------------------------------2-4**|**GTP------------------------------------------------------------------------------------------------------2-4**||
|---|---|---|---|
|**2. **|**GA & SLD ----------------------------------------------------------------------------------------------5-6**|||
|**3. **|**BOM-----------------------------------------------------------------------------------------------------7**|||
|**4.**|Tpsperature Rise----------------------------------------------------------------------------------8|||
|**5. **|**SCB Type Test Report------------------------------------------------------------------------------9-98**|||
|**6. **|**Bus Bar Calculation---------------------------------------------------------------------------------99-101**|

#### 11. Side-by-Side Comparison

In [4]:
raw_text = page.get_text()

markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[0]
)

print("=" * 80)
print("RAW TEXT")
print("=" * 80)
print(raw_text[:3000])

print("=" * 80)
print("MARKDOWN")
print("=" * 80)
print(markdown[:3000])

=== Document parser messages ===
                                    Using Tesseract for OCR processing.

RAW TEXT
 
 
 
. 
 
Table Of Content 
 
1.  GTP------------------------------------------------------------------------------------------------------2-4 
2. GA & SLD ----------------------------------------------------------------------------------------------5-6 
3. BOM-----------------------------------------------------------------------------------------------------7 
4. 
5. SCB Type Test Report------------------------------------------------------------------------------9-98 
6. Bus Bar Calculation---------------------------------------------------------------------------------99-101 
7. 
8. SPD Data Sheet & Certificate---------------------------------------------------------------------120-127 
9. Polyamide Gland- Data Sheet & Certificate--------------------------------------------------128-133 
10.
11. Solar Cable- Data Sheet & Certificate------------------------------------

For learning RAG ingestion, I would organize the notebook as a **progressive investigation**, not as a collection of random API calls.

The goal is:

```text
PDF
 ↓
How does PyMuPDF see the document?
 ↓
Where does extraction fail?
 ↓
How does PyMuPDF4LLM improve it?
 ↓
How would I build a production ingestion pipeline?
```

# Notebook Structure

---

# 0. Setup

```python
import fitz
import pymupdf
import pymupdf4llm

PDF_PATH = "../data/Harwick_Technical_Documents.pdf"

print("PyMuPDF:", pymupdf.__version__)

doc = fitz.open(PDF_PATH)

print("Pages:", doc.page_count)
```

---

# 1. Visual Inspection First

Before extracting anything, look at the page.

```python
page = doc[0]

pix = page.get_pixmap(dpi=200)

pix.save("page_1.png")

print("Saved page_1.png")
```

Questions:

* Single column?
* Two columns?
* Tables?
* Images?
* Headers/footers?
* Scanned PDF?

You should always answer these first.

---

# 2. Raw Text Extraction

Most basic extraction.

```python
page = doc[0]

text = page.get_text()

print(text[:5000])
```

Observe:

* Is text missing?
* Is ordering wrong?
* Are tables flattened?

---

# 3. Understand PDF Blocks

A PDF is not text.

A PDF is positioned objects.

```python
blocks = page.get_text("blocks")

print("Block count:", len(blocks))
```

---

Inspect blocks:

```python
for idx, block in enumerate(blocks):

    print("\n" + "=" * 80)
    print(f"BLOCK {idx}")
    print("=" * 80)

    print("BBox:", block[:4])
    print("Type:", block[6])

    print(block[4][:500])
```

Learn:

```text
Block
 ├── Bounding Box
 ├── Type
 └── Content
```

This is where reading-order problems start.

---

# 4. Reading Order Reconstruction

PDFs don't guarantee order.

PyMuPDF gives coordinates.

Try sorting.

```python
blocks = page.get_text("blocks")

blocks = sorted(
    blocks,
    key=lambda b: (b[1], b[0])
)

for block in blocks:
    print(block[4])
```

Compare:

```text
Original order
vs
Sorted order
```

This is exactly what your production extractor is doing.

---

# 5. Deep Dive Into Structure

Most important cell.

```python
content = page.get_text("dict")
```

Inspect:

```python
from pprint import pprint

pprint(content["blocks"][0])
```

Understand hierarchy:

```text
Page
 └── Block
      └── Line
            └── Span
```

Example:

```python
for block in content["blocks"]:

    if "lines" not in block:
        continue

    for line in block["lines"]:

        for span in line["spans"]:

            print(span["text"])
            print(span["font"])
            print(span["size"])
```

This is how you detect:

* headings
* bold text
* section titles

---

# 6. Word Level Extraction

Useful for table reconstruction.

```python
words = page.get_text("words")

for word in words[:20]:
    print(word)
```

Structure:

```text
(
 x0,
 y0,
 x1,
 y1,
 word,
 block,
 line,
 word_no
)
```

Observe coordinates.

---

# 7. Image Analysis

Inspect embedded images.

```python
images = page.get_images(full=True)

print("Images:", len(images))
```

---

Extract images.

```python
for idx, image in enumerate(images):

    xref = image[0]

    pix = fitz.Pixmap(doc, xref)

    output = f"image_{idx}.png"

    if pix.alpha:
        pix = fitz.Pixmap(fitz.csRGB, pix)

    pix.save(output)

    print(output)
```

Questions:

* Logos?
* Diagrams?
* Scanned content?

---

# 8. Table Detection

Critical for procurement docs.

```python
tables = page.find_tables()

print("Detected Tables:", len(tables.tables))
```

---

Inspect tables.

```python
for idx, table in enumerate(tables.tables):

    print("\nTABLE", idx)

    data = table.extract()

    for row in data:
        print(row)
```

Observe:

* Missing rows?
* Broken columns?
* Merged cells?

---

# 9. Compare Raw Text vs Tables

Very important.

```python
print(page.get_text()[:3000])
```

Compare with:

```python
table.extract()
```

You'll quickly see why tables need special handling.

---

# 10. PyMuPDF4LLM

Now compare against the LLM-friendly version.

```python
markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[0]
)

print(markdown)
```

Observe:

* Headings
* Lists
* Tables
* Reading order

---

# 11. Side-by-Side Comparison

```python
raw_text = page.get_text()

markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[0]
)

print("=" * 80)
print("RAW TEXT")
print("=" * 80)
print(raw_text[:3000])

print("=" * 80)
print("MARKDOWN")
print("=" * 80)
print(markdown[:3000])
```

This is the most important comparison.

---

# 12. RAG Perspective

Create a summary table.

| Extraction Method | Best For             | Problems             |
| ----------------- | -------------------- | -------------------- |
| get_text()        | Simple PDFs          | Reading order issues |
| blocks            | Layout analysis      | Needs custom logic   |
| dict              | Advanced parsing     | More complex         |
| words             | Table reconstruction | Low-level            |
| find_tables()     | Structured tables    | Not perfect          |
| pymupdf4llm       | RAG ingestion        | Less control         |

---

# 13. Production Reflection

After every PDF answer:

```text
1. Is it digital or scanned?
2. Did get_text() work?
3. Did reading order break?
4. Were tables extracted correctly?
5. Did pymupdf4llm improve output?
6. Do I need OCR?
7. What chunking strategy would I use?
```

That final step is where the notebook becomes a **RAG ingestion learning notebook**, not just a PyMuPDF API tutorial.

For your RAG evaluation journey, this notebook is valuable because it teaches the root cause of many retrieval failures: the problem is often not embeddings or retrieval at all—it starts with poor extraction before chunking ever happens.


In [ ]:
markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[1]
)

print(markdown)

In [ ]:
markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[2]
)

print(markdown)

In [ ]:
markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[4]
)

print(markdown)

In [ ]:
markdown = pymupdf4llm.to_markdown(
    PDF_PATH,
    pages=[6]
)

print(markdown)

In [ ]:
from docling.document_converter import DocumentConverter

converter = DocumentConverter()

In [ ]:
result = converter.convert(PDF_PATH,page_range=(4, 4))

print(result.document.export_to_markdown())